In [ ]:
# Make sure you are in your deep learning environment (with PyTorch and Transformers)

import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import wandb
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import matplotlib.pyplot as plt

# ==========================================
# PHASE 1: Data Acquisition
# ==========================================
print("📥 Loading 200k production dataset from Hugging Face...")
hf_dataset = load_dataset("lanretto/discourse_quality", data_files="full_labeled_flattened.csv", split="train")
df = hf_dataset.to_pandas()

target_cols = [
    'level_of_justification', 'respect_towards_demands', 'respect_towards_counterarguments',
    'content_of_justification', 'respect_towards_groups', 'constructive_politics'
]
df_model = df[['comment'] + target_cols].dropna()
df_model['comment'] = df_model['comment'].astype(str)

print("🔄 'Zipping' dimensions and splitting data...")
df_model['labels'] = df_model[target_cols].values.tolist()
train_df, test_df = train_test_split(df_model, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df[['comment', 'labels']], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[['comment', 'labels']], preserve_index=False)

# ==========================================
# PHASE 2: Tokenization (Swapped to RoBERTa!)
# ==========================================
print("⚙️ Loading the roberta-base Tokenizer...")
# CHANGE 1: We tell AutoTokenizer to pull RoBERTa instead of BERT
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_function(examples):
    return tokenizer(examples['comment'], padding="max_length", truncation=True, max_length=128)

print("Tokenizing data...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# ==========================================
# PHASE 3: Model Architecture (Swapped to RoBERTa!)
# ==========================================
print("🧠 Loading the RoBERTa Master Model...")
# CHANGE 2: We tell AutoModel to pull RoBERTa instead of BERT
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=6,
    problem_type="regression"
)

# ==========================================
# PHASE 4: Validation Grader
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    metrics = {}
    total_mse = 0
    for i, col_name in enumerate(target_cols):
        mse = mean_squared_error(labels[:, i], logits[:, i])
        metrics[f"mse_{col_name}"] = mse
        total_mse += mse
    metrics["mse_OVERALL"] = total_mse / 6
    return metrics

# ==========================================
# PHASE 5: W&B Integration & Training
# ==========================================
print("🔗 Connecting to Weights & Biases...")
os.environ["WANDB_API_KEY"] = "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"

wandb.init(
    entity="niyimoluga85-universty-of-technology-graz",
    project="DQI_Multi_Regression",
    name="roberta_base_6_dim", # CHANGE 3: Updated the run name for tracking!
)

training_args = TrainingArguments(
    output_dir="./results_roberta_master",
    eval_strategy="steps",
    logging_strategy="steps",
    save_strategy="steps",
    eval_steps=200,
    logging_steps=200,
    save_steps=200,
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=64, # Keeping your massive batch size!
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="mse_OVERALL",
    greater_is_better=False,
    report_to="wandb"
)

class PureMSETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.MSELoss(reduction="none")
        per_dim_errors = loss_fct(logits, labels)
        mse_per_dim = per_dim_errors.mean(dim=0)
        master_mse = mse_per_dim.mean()
        
        if self.state.global_step % self.args.logging_steps == 0 and self.args.should_log:
            log_dict = {}
            for i, col_name in enumerate(target_cols):
                log_dict[f"train/mse_{col_name}"] = mse_per_dim[i].item()
            log_dict["train/mse_OVERALL"] = master_mse.item()
            self.log(log_dict) # Fixed typo from previous version!
            
        return (master_mse, outputs) if return_outputs else master_mse

trainer = PureMSETrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("\n🚀 Starting RoBERTa Multi-Output Training...")
trainer.train()
wandb.finish()
print("\n✅ Training Complete!")